In [1]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from tqdm import tqdm
import os
from scipy.stats import norm
from scipy.special import ndtr
from pathlib import Path

In [2]:
ROOT = Path.cwd().parent
DATA = ROOT / "data"
RESULTS = ROOT / "results"

In [3]:
names = locals()

In [4]:
df_cement_production          = pd.read_csv(DATA / 'production-142.csv', index_col=0)
economies_142                 = df_cement_production.columns.tolist()
ssp = 'SSP2'
stock_growth_of_economies_142 = pd.read_excel(RESULTS / 'future-stock-growth.xlsx', sheet_name=ssp, index_col=0)

region_country = pd.read_excel(DATA / 'contents.xlsx', sheet_name='region_division')
region_country.loc[28, 'region'] = 'NA'
region_country.loc[171, 'region'] = 'NA'

In [5]:
def make_uncertain_values(df, std, span, run, seed):
    values = np.array([df]) if type(df) == float else np.array(df)
    if len(values) == 92:
        np.random.seed(seed)
        gg = np.random.normal(values, values * std, size=(run, span))
    elif len(values) == 1:
        np.random.seed(seed)
        gg = np.random.normal(values, values * std, size=(run, span))

    return gg.astype(np.float32) 

In [6]:
def fit_logistic_with_constraints(
                                  hist_values,
                                  future_growth,
                                  start=1931, mid=2022, end=2050,
                                    r_grid=np.linspace(0.05, 0.8, 100),
                                    t0_grid=np.linspace(1950, 2040, 120),
                                    recent_weight=3.0, recent_from=1990):
        
    def logistic_4p(t, A, K, r, t0):
        return A + (K - A) / (1.0 + np.exp(-r * (t - t0)))
    
    def solve_AK_for_endpoints(r, t0, y_mid, y_end):
        L1 = 1.0 / (1.0 + np.exp(-r * (mid - t0)))
        L2 = 1.0 / (1.0 + np.exp(-r * (end - t0)))
        M = np.array([[1 - L1, L1],
                    [1 - L2, L2]], dtype=float)
        y = np.array([y_mid, y_end], dtype=float)
        A, K = np.linalg.solve(M, y)
        return A, K


    hist_years = np.array([t for t in range(start, mid+1)])
    # hist_values = np.array(df_hist.values.tolist())

    y_mid = hist_values[-1]
    y_end = y_mid * future_growth

    w = np.ones_like(hist_years, dtype=float)
    w[hist_years >= recent_from] = recent_weight

    best = {"err": np.inf, "r": None, "t0": None, "A": None, "K": None}
    for r in r_grid:
        for t0 in t0_grid:
            L1 = 1.0 / (1.0 + np.exp(-r * (mid - t0)))
            L2 = 1.0 / (1.0 + np.exp(-r * (end - t0)))
            if abs(L2 - L1) < 1e-6:
                continue

            A, K = solve_AK_for_endpoints(r, t0, y_mid, y_end)
            if not (r > 0 and K > A):
                continue

            y_hat = logistic_4p(hist_years, A, K, r, t0)
            err = np.average((y_hat - hist_values) ** 2, weights=w)
            if err < best["err"]:
                best = {"err": err, "r": r, "t0": t0, "A": A, "K": K}


    r = best["r"]
    t0 = best["t0"]
    A = best["A"]
    K = best["K"]
    years = np.arange(start, end + 1, dtype=float)
    values = logistic_4p(years, A, K, r, t0)
    df_results = pd.Series(values, years)

    return df_results

In [7]:
def cal_dem_rate_matrix(lft_mat, start=1931, mid=2022, end=2050):
    """
    优化点：
    - 不再 np.repeat 物化 lft_3D（节省 ~100MB 分配 + 写回）
    - norm.cdf → ndtr：z = 5*(age/lft - 1)，避开 scipy 包装层
    - 去掉 gg 的 28 次 tile，直接广播 (n_sim, n_hist, 1) 的 denom
    """
    n_sim, n_hist = lft_mat.shape
    span = end - mid + 1

    age_2D = np.array([[j - i for j in range(mid, end + 1)] for i in range(start, mid + 1)],
                      dtype=np.float32)                              # (92, 29)

    # z = (age - lft)/(lft*0.2) = 5*(age/lft - 1)，全程广播
    lft_3D_view = lft_mat[:, :, None]                                # (n_sim, 92, 1)
    z = 5.0 * (age_2D[None, :, :] / lft_3D_view - 1.0)               # (n_sim, 92, 29)
    ddd = ndtr(z).astype(np.float32)

    denom = 1.0 - ddd[:, :, [0]]                                     # (n_sim, 92, 1)，沿年份恒定
    mask = denom == 0
    safe_denom = np.where(mask, 1.0, denom)

    sur_rate = np.where(mask, 0.0, (1.0 - ddd[:, :, 1:]) / safe_denom)
    dem_rate = np.where(mask, 0.0, np.diff(ddd, axis=2) / safe_denom)

    return sur_rate, dem_rate

In [8]:
def cal_dem_rate_matrix2(fut_lft_mat, mid=2022, end=2050):
    """
    优化点：
    - fut_lft_mat 是 lft[:, -1]*ext 沿 axis=1 复制 span 次得到的，每列相同 → 用 1D 向量
    - 不再 np.repeat 物化 lft_3D ((n_sim, 28, 28) 冗余)
    - norm.cdf → ndtr
    NaN 语义保持不变。
    """
    span = end - mid
    n_sim = fut_lft_mat.shape[0]

    lft_1D = fut_lft_mat[:, 0]                                       # (n_sim,) — 每列相同，取一列即可

    # age_2D[j, i] = i - j；上三角（i<j）置 NaN
    i_idx = np.arange(span, dtype=np.float32)[None, :]
    j_idx = np.arange(span, dtype=np.float32)[:, None]
    age_2D = i_idx - j_idx
    age_2D = np.where(age_2D >= 0, age_2D, np.nan)                   # (span, span)

    z = 5.0 * (age_2D[None, :, :] / lft_1D[:, None, None] - 1.0)     # (n_sim, span, span)
    ccc = ndtr(z).astype(np.float32)                                 # NaN 自然传播

    sur_rate = 1.0 - ccc

    dem_rate = np.full(sur_rate.shape, np.nan, dtype=np.float32)
    dem_rate[:, :, 1:] = np.diff(ccc, axis=2)
    # 对角线：新建当年的拆除率
    diag = np.arange(span)
    dem_rate[:, diag, diag] = ccc[:, diag, diag]

    return sur_rate, dem_rate

In [9]:
def process_a_flow(
        economy,
        material, 
        app, 
        strength, 
        sur_rate, 
        dem_rate, 
        fut_sur_rate, 
        fut_dem_rate,
        future_stock_trajectory,
        hist_dir,
        output_dir, 
        n_sim=10000,
        seed=1
        ):

    mid, end = 2022, 2050
    span = end - mid

    # lft = np.vstack([data1[f'lft_{app}'], data2[f'lft_{app}']])
    # fut_lft = np.array([lft[:, -1] * lft_extension for i in range(span)]).T
    # sur_rate, dem_rate = cal_dem_rate_matrix(lft)
    # fut_sur_rate, fut_dem_rate = cal_dem_rate_matrix2(fut_lft)

    if_developed = region_country[region_country['ISO-3'] == economy]['developed'].values[0]
    # std = 0.05 if if_developed == 1 else 0.1
    std = 0.05

    coarse_to_binder_params = {'C15': 4.182638806, 'C16_23': 3.271740668, 'C24_35': 2.584728036, 'C35': 2.471397967}
    fine_to_binder_params   = {'C15': 2.542840011, 'C16_23': 2.855202053, 'C24_35': 2.083908811, 'C35': 1.728617340}

    if material == 'concrete':
        stock_cohort     = np.load(hist_dir / f'S_binder_for_concrete_{app}_{strength}_cohort.npz')['data']
        his_stock_annual = np.load(hist_dir / f'S_binder_for_concrete_{app}_{strength}_annual.npz')['data']

        alpha = coarse_to_binder_params[strength]
        beta  = fine_to_binder_params[strength]

        coarse_to_binder = make_uncertain_values(alpha, std, span, n_sim, seed)
        fine_to_binder   = make_uncertain_values(beta,  std, span, n_sim, seed)

    else:
        stock_cohort     = np.load(hist_dir / f'S_binder_for_mortar_{app}_cohort.npz')['data']
        his_stock_annual = np.load(hist_dir / f'S_binder_for_mortar_{app}_annual.npz')['data']
        fine_to_binder   = make_uncertain_values(2.720389651, std, span, n_sim, seed)

    fut_his_stock_array_ = np.array([stock_cohort for i in range(span)]).transpose(1, 2, 0)
    fut_his_stock_array = fut_his_stock_array_ * sur_rate
    fut_his_outflow_array = fut_his_stock_array_ * dem_rate
    fut_his_stock_annual = np.nansum(fut_his_stock_array, axis=1)
    fut_his_outflow_annual = np.nansum(fut_his_outflow_array, axis=1)

    current_stock = his_stock_annual[:, -1]
    future_stock_annual = current_stock[:, None] * future_stock_trajectory[None, :]

    the_gap = future_stock_annual[:, 1:] - fut_his_stock_annual

    fut_fut_stock_array = np.full(shape=(n_sim, span, span), fill_value=np.nan).astype(np.float32)
    fut_fut_outflow_array = np.full(shape=(n_sim, span, span), fill_value=np.nan).astype(np.float32)
    future_inflows = np.empty((span, n_sim), dtype=np.float32)
    col_sums = np.zeros((n_sim, span), dtype=np.float32)

    for t in range(span):
        a_inflow = the_gap[:, t] - col_sums[:, t]
        row_stock   = a_inflow[:, None] * fut_sur_rate[:, t, :]          # 上三角为 NaN，下三角有效
        row_outflow = a_inflow[:, None] * fut_dem_rate[:, t, :]
        fut_fut_stock_array[:, t, :]   = row_stock
        fut_fut_outflow_array[:, t, :] = row_outflow
        future_inflows[t] = a_inflow
        # NaN 视为 0 累加（fut_sur_rate 的 NaN 不该污染列累加）
        col_sums += np.nan_to_num(row_stock, nan=0.0)
    
    fut_fut_stock_annual   = np.nansum(fut_fut_stock_array, axis=1)
    fut_fut_outflow_annual = np.nansum(fut_fut_outflow_array, axis=1)

    future_inflows_annual  = future_inflows.T
    future_outflows_annual = fut_his_outflow_annual + fut_fut_outflow_annual

    # future_fine_inflow = future_inflows_annual * fine_to_binder[:, None]
    # future_fine_outflow = future_outflows_annual * fine_to_binder[:, None]

    future_fine_inflow  = future_inflows_annual * fine_to_binder
    future_fine_outflow = future_outflows_annual * fine_to_binder


    if material == 'concrete':
        # print(future_inflows_annual.shape)
        # futuer_coarse_inflow = future_inflows_annual * coarse_to_binder[:, None]
        # future_coarse_outflow = future_outflows_annual * coarse_to_binder[:, None]
        futuer_coarse_inflow = future_inflows_annual * coarse_to_binder
        future_coarse_outflow = future_outflows_annual * coarse_to_binder

        np.savez_compressed(os.path.join(output_dir, f'S_binder_for_concrete_{app}_{strength}_his_annual.npz'), data=fut_his_stock_annual)
        np.savez_compressed(os.path.join(output_dir, f'O_binder_for_concrete_{app}_{strength}_his_annual.npz'), data=fut_his_outflow_annual)
        np.savez_compressed(os.path.join(output_dir, f'S_binder_for_concrete_{app}_{strength}_annual.npz'), data=future_stock_annual)
        np.savez_compressed(os.path.join(output_dir, f'I_binder_for_concrete_{app}_{strength}_annual.npz'), data=future_inflows_annual)
        np.savez_compressed(os.path.join(output_dir, f'O_binder_for_concrete_{app}_{strength}_annual.npz'), data=future_outflows_annual)
        np.savez_compressed(os.path.join(output_dir, f'I_coarse_for_concrete_{app}_{strength}_annual.npz'), data=futuer_coarse_inflow)
        np.savez_compressed(os.path.join(output_dir, f'O_coarse_for_concrete_{app}_{strength}_annual.npz'), data=future_coarse_outflow)
        np.savez_compressed(os.path.join(output_dir, f'I_fine_for_concrete_{app}_{strength}_annual.npz'), data=future_fine_inflow)
        np.savez_compressed(os.path.join(output_dir, f'O_fine_for_concrete_{app}_{strength}_annual.npz'), data=future_fine_outflow)
        
    else:
        np.savez_compressed(os.path.join(output_dir, f'S_binder_for_mortar_{app}_his_annual.npz'), data=fut_his_stock_annual)
        np.savez_compressed(os.path.join(output_dir, f'O_binder_for_mortar_{app}_his_annual.npz'), data=fut_his_outflow_annual)
        np.savez_compressed(os.path.join(output_dir, f'S_binder_for_mortar_{app}_annual.npz'), data=future_stock_annual)
        np.savez_compressed(os.path.join(output_dir, f'I_binder_for_mortar_{app}_annual.npz'), data=future_inflows_annual)
        np.savez_compressed(os.path.join(output_dir, f'O_binder_for_mortar_{app}_annual.npz'), data=future_outflows_annual)
        np.savez_compressed(os.path.join(output_dir, f'I_fine_for_mortar_{app}_annual.npz'), data=future_fine_inflow)
        np.savez_compressed(os.path.join(output_dir, f'O_fine_for_mortar_{app}_annual.npz'), data=future_fine_outflow)

In [10]:
variable_list = [
    'I_binder_annual',
    'I_fine_annual',
    'I_coarse_annual',
    'O_binder_annual',
    'O_fine_annual',
    'O_coarse_annual',
    'reuse_factors',
    'recycle_factors',
    'sand_sub_factors',
    'fut_CCR',
    'I_binder_for_concrete_annual_ru',
    'I_binder_for_mortar_annual_ru',
    'I_binder_for_concrete_annual_v',
    'I_binder_for_mortar_annual_v',
    'I_binder_annual_ru',
    'I_binder_annual_v',
    'p_binder_slag',
    'p_binder_FAP',
    'p_binder_in_exportPro',
    'p_binder_in_importPro',
    'p_cement_export',
    'p_cement_import',
    'p_clinker_export',
    'p_clinker_import',
    'p_sand_export',
    'p_sand_import',
    'p_gravel_export',
    'p_gravel_import',
    'p_limestone_export',
    'p_limestone_import',
    'gravel_export',
    'gravel_import',
    'sand_export',
    'sand_import',
    'excess_gravel',
    'excess_sand',
    'gravel_stock',
    'sand_stock',
    'consumed_gravel_stock',
    'consumed_sand_stock',
    'gravel_production',
    'sand_production',
    'reused_and_recycled_gravel',
    'p_reused_gravel',
    'p_recycled_gravel',
    'reused_gravel',
    'recycled_gravel',
    'gravel_extraction',
    'reused_and_recycled_sand',
    'p_reused_sand',
    'p_recycled_sand',
    'reused_sand',
    'recycled_sand',
    'substituted_sand',
    'sand_extraction',
    'all_binder',
    'cement_consumption',
    'cement_production',
    'clinker_consumption',
    'excess_clinker',
    'clinker_stock',
    'clinker_production',
    'r_Ca',
    'RawMat_Ca',
    'limestone_export',
    'limestone_import',
    'excess_limestone',
    'limestone_stock',
    'consumed_limestone_stock',
    'limestone_stock',
    'limestone_extraction',
    'S_coarse_annual',
    'S_fine_annual',
    'S_binder_annual',
    'split_ratio_coarse',
    'split_ratio_fine',
    'split_ratio_RawMat_Ca',
    'gravel_extraction_for_stock_growth',
    'gravel_extraction_for_stock_turnover',
    'sand_extraction_for_stock_growth',
    'sand_extraction_for_stock_turnover',
    'limestone_extraction_for_stock_growth',
    'limestone_extraction_for_stock_turnover'
]

In [11]:
def run_prediction(
    scenario,
    economy,
    more_intensive_use=0,
    lifetime_extension=0,
    reuse_rate=0,
    recycle_rate=0,
    sand_sub=0,
    CCR_reduction=0,
    end=2050,
    mid=2022,
    n_sim=10000,
    seed=1
):
    
    hist_dir   = RESULTS / 'material-flows' / economy / 'historical'
    output_dir = RESULTS / 'material-flows' / economy / scenario
    output_dir.mkdir(parents=True, exist_ok=True)

    if_developed = region_country[region_country['ISO-3'] == economy]['developed'].values[0]
    std = 0.05 if if_developed == 1 else 0.1
    
    lft_extension = 1 + lifetime_extension
    span = end - mid

    S_binder_annual = np.load(hist_dir / 'S_binder_annual.npz')['data']

    historical_stock_trajectory = np.median(S_binder_annual, axis=0) / np.median(S_binder_annual, axis=0)[-1]
    K = stock_growth_of_economies_142.loc[end, economy]
    fitted_stock_trajectory = fit_logistic_with_constraints(historical_stock_trajectory, K)
    future_stock_trajectory_ = fitted_stock_trajectory.loc[mid:end].values
    fut_stock_factor = np.linspace(1, 1 - more_intensive_use, span+1)
    future_stock_trajectory = future_stock_trajectory_ * fut_stock_factor

    for app in ['RES', 'NRES', 'CIV']:
        lft = np.load(hist_dir / f'lft_{app}.npz')['data']
        fut_lft = np.array([lft[:, -1] * lft_extension for i in range(span)]).T
        sur_rate, dem_rate = cal_dem_rate_matrix(lft)
        fut_sur_rate, fut_dem_rate = cal_dem_rate_matrix2(fut_lft)

        for strength in ['C15', 'C16_23', 'C24_35', 'C35']:
            process_a_flow(
                economy,
                'concrete', 
                app, 
                strength,
                sur_rate,
                dem_rate,
                fut_sur_rate,
                fut_dem_rate,
                future_stock_trajectory,
                hist_dir,
                output_dir)

        binder_stock = np.nansum([np.load(output_dir / f'S_binder_for_concrete_{app}_{strength}_annual.npz')['data'] for strength in ['C15', 'C16_23', 'C24_35', 'C35']], axis=0)
        binder_inflow = np.nansum([np.load(output_dir / f'I_binder_for_concrete_{app}_{strength}_annual.npz')['data'] for strength in ['C15', 'C16_23', 'C24_35', 'C35']], axis=0)
        binder_outflow = np.nansum([np.load(output_dir / f'O_binder_for_concrete_{app}_{strength}_annual.npz')['data'] for strength in ['C15', 'C16_23', 'C24_35', 'C35']], axis=0)
        coarse_inflow = np.nansum([np.load(output_dir / f'I_coarse_for_concrete_{app}_{strength}_annual.npz')['data'] for strength in ['C15', 'C16_23', 'C24_35', 'C35']], axis=0)
        coarse_outflow = np.nansum([np.load(output_dir / f'O_coarse_for_concrete_{app}_{strength}_annual.npz')['data'] for strength in ['C15', 'C16_23', 'C24_35', 'C35']], axis=0)
        fine_inflow = np.nansum([np.load(output_dir / f'I_fine_for_concrete_{app}_{strength}_annual.npz')['data'] for strength in ['C15', 'C16_23', 'C24_35', 'C35']], axis=0)
        fine_outflow = np.nansum([np.load(output_dir / f'O_fine_for_concrete_{app}_{strength}_annual.npz')['data'] for strength in ['C15', 'C16_23', 'C24_35', 'C35']], axis=0)

        np.savez_compressed(output_dir / f'S_binder_for_concrete_{app}_annual.npz', data=binder_stock)
        np.savez_compressed(output_dir / f'I_binder_for_concrete_{app}_annual.npz', data=binder_inflow)
        np.savez_compressed(output_dir / f'O_binder_for_concrete_{app}_annual.npz', data=binder_outflow)
        np.savez_compressed(output_dir / f'I_coarse_for_concrete_{app}_annual.npz', data=coarse_inflow)
        np.savez_compressed(output_dir / f'O_coarse_for_concrete_{app}_annual.npz', data=coarse_outflow)
        np.savez_compressed(output_dir / f'I_fine_for_concrete_{app}_annual.npz',   data=fine_inflow)
        np.savez_compressed(output_dir / f'O_fine_for_concrete_{app}_annual.npz',   data=fine_outflow)

        process_a_flow(
            economy,
            'mortar', 
            app,
            '',
            sur_rate,
            dem_rate,
            fut_sur_rate,
            fut_dem_rate,
            future_stock_trajectory,
            hist_dir,
            output_dir
            )

    for material in ['concrete', 'mortar']:
        binder_stock = np.nansum([np.load(output_dir / f'S_binder_for_{material}_{app}_annual.npz')['data'] for app in ['RES', 'NRES', 'CIV']], axis=0).astype(np.float32)
        binder_inflow = np.nansum([np.load(output_dir / f'I_binder_for_{material}_{app}_annual.npz')['data'] for app in ['RES', 'NRES', 'CIV']], axis=0).astype(np.float32)
        binder_outflow = np.nansum([np.load(output_dir / f'O_binder_for_{material}_{app}_annual.npz')['data'] for app in ['RES', 'NRES', 'CIV']], axis=0).astype(np.float32)
        fine_inflow = np.nansum([np.load(output_dir / f'I_fine_for_{material}_{app}_annual.npz')['data'] for app in ['RES', 'NRES', 'CIV']], axis=0).astype(np.float32)
        fine_outflow = np.nansum([np.load(output_dir / f'O_fine_for_{material}_{app}_annual.npz')['data'] for app in ['RES', 'NRES', 'CIV']], axis=0).astype(np.float32)
        
        np.savez_compressed(output_dir / f'S_binder_for_{material}_annual.npz', data=binder_stock)
        np.savez_compressed(output_dir / f'I_binder_for_{material}_annual.npz', data=binder_inflow)
        np.savez_compressed(output_dir / f'O_binder_for_{material}_annual.npz', data=binder_outflow)
        np.savez_compressed(output_dir / f'I_fine_for_{material}_annual.npz',   data=fine_inflow)
        np.savez_compressed(output_dir / f'O_fine_for_{material}_annual.npz',   data=fine_outflow)

        if material == 'concrete':
            coarse_inflow  = np.nansum([np.load(output_dir / f'I_coarse_for_{material}_{app}_annual.npz')['data'] for app in ['RES', 'NRES', 'CIV']], axis=0).astype(np.float32)
            coarse_outflow = np.nansum([np.load(output_dir / f'O_coarse_for_{material}_{app}_annual.npz')['data'] for app in ['RES', 'NRES', 'CIV']], axis=0).astype(np.float32)

            np.savez_compressed(output_dir / f'I_coarse_for_{material}_annual.npz', data=coarse_inflow)
            np.savez_compressed(output_dir / f'O_coarse_for_{material}_annual.npz', data=coarse_outflow)
    
    I_binder_for_concrete_annual = np.load(output_dir / 'I_binder_for_concrete_annual.npz')['data']
    I_binder_for_mortar_annual =   np.load(output_dir / 'I_binder_for_mortar_annual.npz')['data']
    I_binder_annual = I_binder_for_concrete_annual + I_binder_for_mortar_annual
    I_fine_annual = np.load(output_dir / 'I_fine_for_concrete_annual.npz')['data'] + np.load(output_dir / 'I_fine_for_mortar_annual.npz')['data']
    I_coarse_annual = np.load(output_dir / 'I_coarse_for_concrete_annual.npz')['data']
    O_binder_annual = np.load(output_dir / 'O_binder_for_concrete_annual.npz')['data'] + np.load(output_dir / 'O_binder_for_mortar_annual.npz')['data']
    O_fine_annual = np.load(output_dir / 'O_fine_for_concrete_annual.npz')['data'] + np.load(output_dir / 'O_fine_for_mortar_annual.npz')['data']
    O_coarse_annual = np.load(output_dir / 'O_coarse_for_concrete_annual.npz')['data']

    reuse_factors = np.tile(np.linspace(0, reuse_rate, span+1)[1:], (n_sim, 1))
    recycle_factors = np.tile(np.linspace(0, recycle_rate, span+1)[1:], (n_sim, 1))
    sand_sub_factors = np.tile(np.linspace(0, sand_sub, span+1)[1:], (n_sim, 1))

    CCR_factors = np.tile(np.linspace(1, 1 - CCR_reduction, span+1)[1:], (n_sim, 1))
    his_CCR = np.load(hist_dir / 'CCR.npz')['data']
    fut_CCR = CCR_factors * his_CCR[:, -1][:, None]

    I_binder_for_concrete_annual_ru = np.load(output_dir / 'O_binder_for_concrete_annual.npz')['data'] * reuse_factors 
    I_binder_for_mortar_annual_ru = np.load(output_dir / 'O_binder_for_mortar_annual.npz')['data'] * reuse_factors 
    I_binder_for_concrete_annual_v = I_binder_for_concrete_annual - I_binder_for_concrete_annual_ru 
    I_binder_for_mortar_annual_v = I_binder_for_mortar_annual - I_binder_for_mortar_annual_ru
    I_binder_annual_ru = I_binder_for_concrete_annual_ru + I_binder_for_mortar_annual_ru
    I_binder_annual_v =  I_binder_for_concrete_annual_v + I_binder_for_mortar_annual_v

    p_binder_slag = np.load(hist_dir / 'binder_slag.npz')['data'][:, -1]
    p_binder_FAP  = np.load(hist_dir / 'binder_FAP.npz')['data'][:, -1]

    p_binder_in_exportPro   = np.load(hist_dir / 'binder_in_exportPro.npz')['data'][:, -1] / np.load(hist_dir / 'binder_for_concrete.npz')['data'][:, -1]
    p_binder_in_importPro   = np.load(hist_dir / 'binder_in_importPro.npz')['data'][:, -1] / np.load(hist_dir / 'binder_for_concrete.npz')['data'][:, -1]

    his_cement_consumption  = np.load(hist_dir / 'cement_consumption.npz')['data']
    his_cement_export       = np.load(hist_dir / 'cement_export.npz')['data']
    his_cement_import       = np.load(hist_dir / 'cement_import.npz')['data']

    his_clinker_consumption = np.load(hist_dir / 'clinker_consumption.npz')['data']
    his_clinker_export      = np.load(hist_dir / 'clinker_export.npz')['data']
    his_clinker_import      = np.load(hist_dir / 'clinker_import.npz')['data']

    his_fine_total  = np.load(hist_dir / 'fine_total.npz')['data']
    his_sand_export = np.load(hist_dir / 'sand_export.npz')['data']
    his_sand_import = np.load(hist_dir / 'sand_import.npz')['data']

    his_coarse_total  = np.load(hist_dir / 'coarse_total.npz')['data']
    his_gravel_export = np.load(hist_dir / 'gravel_export.npz')['data']
    his_gravel_import = np.load(hist_dir / 'gravel_import.npz')['data']

    his_RawMat_Ca        = np.load(hist_dir / 'RawMat_Ca.npz')['data']
    his_limestone_export = np.load(hist_dir / 'limestone_export.npz')['data']
    his_limestone_import = np.load(hist_dir / 'limestone_import.npz')['data']

    p_cement_export = np.divide(his_cement_export[:, -1], his_cement_consumption[:, -1], out=np.zeros_like(his_cement_export[:, -1], dtype=np.float32), where=his_cement_consumption[:, -1] != 0)
    p_cement_import = np.divide(his_cement_import[:, -1], his_cement_consumption[:, -1], out=np.zeros_like(his_cement_import[:, -1], dtype=np.float32), where=his_cement_consumption[:, -1] != 0)

    p_clinker_export = np.divide(his_clinker_export[:, -1], his_clinker_consumption[:, -1], out=np.zeros_like(his_clinker_export[:, -1], dtype=np.float32), where=his_clinker_consumption[:, -1] != 0)
    p_clinker_import = np.divide(his_clinker_import[:, -1], his_clinker_consumption[:, -1], out=np.zeros_like(his_clinker_import[:, -1], dtype=np.float32), where=his_clinker_consumption[:, -1] != 0)

    p_sand_export = np.divide(his_sand_export[:, -1], his_fine_total[:, -1], out=np.zeros_like(his_sand_export[:, -1], dtype=np.float32), where=his_fine_total[:, -1] != 0)
    p_sand_import = np.divide(his_sand_import[:, -1], his_fine_total[:, -1], out=np.zeros_like(his_sand_import[:, -1], dtype=np.float32), where=his_fine_total[:, -1] != 0)

    p_gravel_export = np.divide(his_gravel_export[:, -1], his_coarse_total[:, -1], out=np.zeros_like(his_gravel_export[:, -1], dtype=np.float32), where=his_coarse_total[:, -1] != 0)
    p_gravel_import = np.divide(his_gravel_import[:, -1], his_coarse_total[:, -1], out=np.zeros_like(his_gravel_import[:, -1], dtype=np.float32), where=his_coarse_total[:, -1] != 0)

    p_limestone_export = np.divide(his_limestone_export[:, -1], his_RawMat_Ca[:, -1], out=np.zeros_like(his_limestone_export[:, -1], dtype=np.float32), where=his_RawMat_Ca[:, -1] != 0)
    p_limestone_import = np.divide(his_limestone_import[:, -1], his_RawMat_Ca[:, -1], out=np.zeros_like(his_limestone_import[:, -1], dtype=np.float32), where=his_RawMat_Ca[:, -1] != 0)

    gravel_production_ = I_coarse_annual / (1 + p_gravel_export - p_gravel_import)[:, None]
    sand_production_ = I_fine_annual / (1 + p_sand_export - p_sand_import)[:, None]

    gravel_export = I_coarse_annual * p_gravel_export[:, None]
    gravel_import = I_coarse_annual * p_gravel_import[:, None]

    sand_export = I_fine_annual * p_sand_export[:, None]
    sand_import = I_fine_annual * p_sand_import[:, None]

    excess_gravel = np.where(gravel_production_ < 0 , -gravel_production_, 0)
    excess_sand = np.where(sand_production_ < 0 , -sand_production_, 0)

    his_gravel_stock = np.load(hist_dir / 'gravel_stock.npz')['data']
    his_sand_stock = np.load(hist_dir / 'sand_stock.npz')['data']
    his_limestone_stock = np.load(hist_dir / 'limestone_stock.npz')['data']

    gravel_stock = excess_gravel.cumsum(axis=1) + his_gravel_stock[:, -1][:, None]
    sand_stock = excess_sand.cumsum(axis=1) + his_sand_stock[:, -1][:, None]

    consumed_gravel_stock = np.where((gravel_stock > 0) & (gravel_production_ > 0), np.where(gravel_production_ <= gravel_stock, gravel_production_, gravel_stock), 0)
    consumed_sand_stock = np.where((sand_stock > 0) & (sand_production_ > 0), np.where(sand_production_ <= sand_stock, sand_production_, sand_stock), 0)

    gravel_production = np.where(gravel_production_ >= consumed_gravel_stock, gravel_production_ - consumed_gravel_stock, 0)
    sand_production = np.where(sand_production_ >= consumed_sand_stock, sand_production_ - consumed_sand_stock, 0)

    reused_gravel_ = O_coarse_annual * reuse_factors
    recycled_gravel_ = O_coarse_annual * recycle_factors
    reused_and_recycled_gravel_ = reused_gravel_ + recycled_gravel_

    reused_and_recycled_gravel = np.where(gravel_production >= reused_and_recycled_gravel_, reused_and_recycled_gravel_, gravel_production)

    p_reused_gravel = np.divide(reused_gravel_, reused_and_recycled_gravel_, out=np.zeros_like(reused_gravel_, dtype=np.float32), where=reused_and_recycled_gravel_ != 0)
    p_recycled_gravel = np.divide(recycled_gravel_, reused_and_recycled_gravel_, out=np.zeros_like(recycled_gravel_, dtype=np.float32), where=reused_and_recycled_gravel_ != 0)
    reused_gravel = reused_and_recycled_gravel * p_reused_gravel
    recycled_gravel = reused_and_recycled_gravel * p_recycled_gravel
    gravel_extraction = gravel_production - reused_and_recycled_gravel

    reused_sand_ = O_fine_annual * reuse_factors
    recycled_sand_ = O_fine_annual * recycle_factors
    reused_and_recycled_sand_ = reused_sand_ + recycled_sand_ 
    reused_and_recycled_sand = np.where(sand_production >= reused_and_recycled_sand_, reused_and_recycled_sand_, sand_production)

    p_reused_sand = np.divide(reused_sand_, reused_and_recycled_sand_, out=np.zeros_like(reused_sand_, dtype=np.float32), where=reused_and_recycled_sand_ != 0)
    p_recycled_sand = np.divide(recycled_sand_, reused_and_recycled_sand_, out=np.zeros_like(recycled_sand_, dtype=np.float32), where=reused_and_recycled_sand_ != 0)
    reused_sand = reused_and_recycled_sand * p_reused_sand
    recycled_sand = reused_and_recycled_sand * p_recycled_sand
    substituted_sand = (sand_production - reused_and_recycled_sand) * sand_sub_factors
    sand_extraction = sand_production - (reused_and_recycled_sand + substituted_sand)

    all_binder = I_binder_for_concrete_annual_v * (1 - p_binder_in_importPro + p_binder_in_exportPro)[:, None] + I_binder_for_mortar_annual_v
    cement_consumption = all_binder / (1 + p_binder_slag + p_binder_FAP)[:, None]
    cement_production = cement_consumption / (1 - p_cement_export + p_cement_import)[:, None]
    clinker_consumption = cement_production * fut_CCR
    clinker_production_ = clinker_consumption / (1 - p_clinker_export + p_clinker_import)[:, None]
    excess_clinker = np.where(clinker_production_ < 0, -clinker_production_, 0)
    hh = excess_clinker.cumsum(axis=0)
    clinker_stock = hh.copy()
    clinker_stock[3:] = hh[3:] - hh[:-3]
    clinker_production = np.where(clinker_production_ >= clinker_stock, clinker_production_ - clinker_stock, 0)

    # r_Ca = np.vstack([data1['r_Ca'], data2['r_Ca']])[:, -1]
    r_Ca = make_uncertain_values(1.45402, std, span, n_sim, seed)
    RawMat_Ca = clinker_production * r_Ca
    limestone_ = RawMat_Ca / (1 - p_limestone_export + p_limestone_import)[:, None]
    limestone_export = RawMat_Ca * p_limestone_export[:, None]
    limestone_import = RawMat_Ca * p_limestone_import[:, None]
    excess_limestone = np.where(limestone_ < 0, -limestone_, 0)
    limestone_stock = excess_limestone.cumsum(axis=1) + his_limestone_stock[:, -1][:, None]
    consumed_limestone_stock = np.where((limestone_stock > 0) & (limestone_ > 0), np.where(limestone_ <= limestone_stock, limestone_, limestone_stock), 0)
    limestone_stock = limestone_stock - consumed_limestone_stock
    limestone_extraction = np.where(limestone_ >= consumed_limestone_stock, limestone_ - consumed_limestone_stock, 0)

    S_coarse_annual = (np.load(hist_dir / 'S_coarse_annual.npz')['data'][:, -1] * future_stock_trajectory[:, None]).T
    S_fine_annual   = (np.load(hist_dir / 'S_fine_annual.npz')['data'][:, -1] * future_stock_trajectory[:, None]).T
    S_binder_annual = (np.load(hist_dir / 'S_binder_annual.npz')['data'][:, -1] * future_stock_trajectory[:, None]).T

    S_coarse_growth = np.diff(S_coarse_annual, axis=1)
    S_fine_growth = np.diff(S_fine_annual, axis=1)
    S_binder_growth = np.diff(S_binder_annual, axis=1)

    split_ratio_coarse = O_coarse_annual / I_coarse_annual
    split_ratio_fine = O_coarse_annual / I_fine_annual
    split_ratio_RawMat_Ca = O_binder_annual / I_binder_annual

    gravel_extraction_for_stock_growth = (1 - split_ratio_coarse) * gravel_extraction
    sand_extraction_for_stock_growth = (1 - split_ratio_fine) * sand_extraction
    limestone_extraction_for_stock_growth = (1 - split_ratio_RawMat_Ca) * limestone_extraction

    gravel_extraction_for_stock_turnover = split_ratio_coarse * gravel_extraction
    sand_extraction_for_stock_turnover = split_ratio_fine * sand_extraction
    limestone_extraction_for_stock_turnover = split_ratio_RawMat_Ca * limestone_extraction
    
    local_vars = locals()
    for var in variable_list:
        an_array = local_vars[var].astype(np.float32)
        np.savez_compressed(output_dir / f'{var}.npz', data=an_array)

In [12]:
a = 

SyntaxError: invalid syntax (2792523948.py, line 1)

In [14]:
for economy in tqdm(['CHN']):
    run_prediction(
        'BAU', 
        economy, 
        more_intensive_use=0.00,
        lifetime_extension=0.0,
        reuse_rate=0.0,
        recycle_rate=0.0,
        sand_sub=0.0,
        CCR_reduction=0.0
    )

100%|██████████| 1/1 [00:17<00:00, 17.08s/it]


In [ ]:
from joblib import Parallel, delayed

: 

: 

In [ ]:
Parallel(n_jobs=8, backend='loky', verbose=10)(
    delayed(run_prediction)(
        'BAU', economy,
        more_intensive_use=0.0, lifetime_extension=0.0,
        reuse_rate=0.0, recycle_rate=0.0,
        sand_sub=0.0, CCR_reduction=0.0
    )
    for economy in economies_142
)

[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done   2 tasks      | elapsed:   25.3s
[Parallel(n_jobs=8)]: Done   9 tasks      | elapsed:  1.0min
[Parallel(n_jobs=8)]: Done  16 tasks      | elapsed:  1.3min
[Parallel(n_jobs=8)]: Done  25 tasks      | elapsed:  2.6min
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:  3.5min
[Parallel(n_jobs=8)]: Done  45 tasks      | elapsed:  4.5min
[Parallel(n_jobs=8)]: Done  56 tasks      | elapsed:  5.6min
[Parallel(n_jobs=8)]: Done  69 tasks      | elapsed:  7.1min
[Parallel(n_jobs=8)]: Done  82 tasks      | elapsed:  8.5min
[Parallel(n_jobs=8)]: Done  97 tasks      | elapsed: 10.0min
[Parallel(n_jobs=8)]: Done 112 tasks      | elapsed: 11.5min
[Parallel(n_jobs=8)]: Done 142 out of 142 | elapsed: 14.3min finished


[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

: 

: 

In [ ]:
for economy in tqdm(['CHN']):
    run_prediction(
        'BAU', 
        economy, 
        more_intensive_use=0.00,
        lifetime_extension=0.0,
        reuse_rate=0.0,
        recycle_rate=0.0,
        sand_sub=0.0,
        CCR_reduction=0.0
    )

100%|██████████| 1/1 [00:17<00:00, 17.72s/it]


: 

: 

In [ ]:
for economy in tqdm(['CHN']):
    run_prediction(
        'LE', 
        economy, 
        more_intensive_use=0.00,    
        lifetime_extension=0.50,
        reuse_rate=0.0,
        recycle_rate=0.0,
        sand_sub=0.0,
        CCR_reduction=0.0
    )

100%|██████████| 1/1 [00:16<00:00, 16.98s/it]


: 

: 

In [ ]:
for economy in tqdm(['CHN']):
    run_prediction(
        'CE', 
        economy, 
        more_intensive_use=0.05,
        lifetime_extension=0.50,
        reuse_rate=0.1,
        recycle_rate=0.2,
        sand_sub=0.2,
        CCR_reduction=0.25
    )

100%|██████████| 1/1 [00:17<00:00, 17.83s/it]


: 

: 

In [ ]:
for economy in tqdm(economies_142):
    run_prediction(
        'BAU', 
        economy, 
        more_intensive_use=0.00,
        lifetime_extension=0.0,
        reuse_rate=0.0,
        recycle_rate=0.0,
        sand_sub=0.0,
        CCR_reduction=0.0
    )

  0%|          | 0/142 [00:00<?, ?it/s]

: 

: 

In [ ]:
for economy in tqdm(economies_142):
    run_prediction(
        'CE', 
        economy, 
        more_intensive_use=0.05,
        lifetime_extension=0.50,
        reuse_rate=0.1,
        recycle_rate=0.2,
        sand_sub=0.2,
        CCR_reduction=0.25
    )

  0%|          | 0/142 [00:00<?, ?it/s]

: 

: 

In [ ]:
for economy in tqdm(economies_142):
    run_prediction(
        'MIU', 
        economy, 
        more_intensive_use=0.05,
        lifetime_extension=0.0,
        reuse_rate=0.0,
        recycle_rate=0.0,
        sand_sub=0.0,
        CCR_reduction=0.0
    )

  0%|          | 0/142 [00:00<?, ?it/s]

: 

: 

In [ ]:
for economy in tqdm(economies_142):
    run_prediction(
        'LE', 
        economy, 
        more_intensive_use=0.00,    
        lifetime_extension=0.50,
        reuse_rate=0.0,
        recycle_rate=0.0,
        sand_sub=0.0,
        CCR_reduction=0.0
    )

  0%|          | 0/142 [00:02<?, ?it/s]


KeyboardInterrupt: 

: 

: 

In [ ]:
for economy in tqdm(economies_142):
    run_prediction(
        'RE', 
        economy, 
        more_intensive_use=0.00,
        lifetime_extension=0.00,
        reuse_rate=0.1,
        recycle_rate=0.0,
        sand_sub=0.0,
        CCR_reduction=0.0
    )

  0%|          | 0/142 [00:00<?, ?it/s]

: 

: 

In [ ]:
for economy in tqdm(economies_142):
    run_prediction(
        'RC', 
        economy, 
        more_intensive_use=0.00,
        lifetime_extension=0.00,
        reuse_rate=0.0,
        recycle_rate=0.2,
        sand_sub=0.0,
        CCR_reduction=0.0
    )

  0%|          | 0/142 [00:00<?, ?it/s]

: 

: 

In [ ]:
for economy in tqdm(economies_142):
    run_prediction(
        'SS', 
        economy, 
        more_intensive_use=0.00,
        lifetime_extension=0.00,
        reuse_rate=0.0,
        recycle_rate=0.0,
        sand_sub=0.2,
        CCR_reduction=0.0
    )

  0%|          | 0/142 [00:00<?, ?it/s]

: 

: 

In [ ]:
for economy in tqdm(economies_142):
    run_prediction(
        'CCR', 
        economy, 
        more_intensive_use=0.00,
        lifetime_extension=0.00,
        reuse_rate=0.0,
        recycle_rate=0.0,
        sand_sub=0.0,
        CCR_reduction=0.25
    )

  0%|          | 0/142 [00:00<?, ?it/s]

: 

: 